# Lung GMM Interactive Batch Processing

A notebook that loads DICOM CT one case at a time, lets you **manually adjust the ROI (lung region)**, and then runs the GMM analysis.

## Workflow

1. **Cell 1-3**: Initial setup (run only once at the start)
2. **Cell 4**: Load the next patient -> an ROI box appears in Slicer
3. **Adjust the ROI in the Slicer view** -> drag it to enclose only the lung region
4. **Cell 5**: Crop with the ROI -> GMM analysis -> save results
5. Repeat Cell 4-5 for all cases
6. **Cell 6**: Export the aggregated CSV for all cases

## Prerequisites

- 3D Slicer 5.x + SlicerJupyter extension
- SlicerDensityLungSegmentation (runs in fallback mode if absent)

---
## 1. Settings

In [ ]:
# ============================================================
# User settings
# ============================================================
INPUT_DIR = "/path/to/Hosogaya_BAL/ILD Data"
OUTPUT_DIR = "/path/to/CT_results/gmm_interactive"

FALLBACK_MODE = False  # Set to True only if the extension does not work

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

---
## 2. Helper function definitions

In [ ]:
import slicer
import os
import json
import csv
import glob
import time
import numpy as np
from datetime import datetime

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# DICOM database initialization
# ============================================================
def ensure_dicom_database():
    """Initialize the Slicer DICOM database"""
    if not slicer.dicomDatabase or not slicer.dicomDatabase.isOpen:
        db_dir = os.path.join(slicer.app.temporaryPath, "SlicerDICOMDatabase")
        os.makedirs(db_dir, exist_ok=True)
        db_path = os.path.join(db_dir, "ctkDICOM.sql")
        dicomBrowser = slicer.modules.dicom.widgetRepresentation().self()
        dicomBrowser.browserWidget.dicomBrowser.setDatabaseDirectory(db_dir)
        slicer.dicomDatabase.openDatabase(db_path)
    print("DICOM database: ready")


# ============================================================
# Search for DICOM directories
# ============================================================
def find_dicom_patients(input_dir):
    """Search ILD Data for DICOM patient directories"""
    patients = []
    for item in sorted(os.listdir(input_dir)):
        item_path = os.path.join(input_dir, item)
        if not os.path.isdir(item_path):
            continue
        dcmdt_path = os.path.join(item_path, "DCMDT")
        if os.path.isdir(dcmdt_path):
            n_files = len([f for f in os.listdir(dcmdt_path)
                           if os.path.isfile(os.path.join(dcmdt_path, f))
                           and not f.startswith('.')])
            if n_files > 0:
                patients.append({
                    "name": item,
                    "path": dcmdt_path,
                    "n_slices": n_files
                })
    return patients


# ============================================================
# Load volume from DICOM
# ============================================================
def load_volume_from_dicom(dicom_dir):
    """Load a volume node from a DICOM directory"""
    from DICOMLib import DICOMUtils

    DICOMUtils.importDicom(dicom_dir)
    slicer.app.processEvents()

    db = slicer.dicomDatabase
    patient_uids = db.patients()
    if not patient_uids:
        return None

    # Collect series within the target directory
    dicom_dir_abs = os.path.abspath(dicom_dir)
    candidate_series = []

    for patient_uid in patient_uids:
        for study_uid in db.studiesForPatient(patient_uid):
            for series_uid in db.seriesForStudy(study_uid):
                files = db.filesForSeries(series_uid)
                if not files:
                    continue
                if os.path.abspath(files[0]).startswith(dicom_dir_abs):
                    candidate_series.append({
                        "series_uid": series_uid,
                        "n_files": len(files)
                    })

    if not candidate_series:
        latest_patient = patient_uids[-1]
        for study_uid in db.studiesForPatient(latest_patient):
            for series_uid in db.seriesForStudy(study_uid):
                files = db.filesForSeries(series_uid)
                if files:
                    candidate_series.append({
                        "series_uid": series_uid,
                        "n_files": len(files)
                    })

    if not candidate_series:
        return None

    # Select the series with the most slices
    candidate_series.sort(key=lambda x: x["n_files"], reverse=True)
    best = candidate_series[0]
    print(f"  Series: {best['series_uid']} ({best['n_files']} slices)")

    loaded_ids = DICOMUtils.loadSeriesByUID([best["series_uid"]])
    slicer.app.processEvents()

    if not loaded_ids:
        return None

    for node_id in loaded_ids:
        node = slicer.mrmlScene.GetNodeByID(node_id)
        if node and node.IsA("vtkMRMLScalarVolumeNode"):
            return node

    return slicer.mrmlScene.GetFirstNodeByClass("vtkMRMLScalarVolumeNode")


# ============================================================
# Create and display the ROI
# ============================================================
def create_lung_roi(volume_node):
    """
    Create an initial ROI based on the volume bounds.
    The Z axis is set to the central 80% (approximate apex-to-base range).
    The user drags it in the Slicer view to adjust.
    """
    bounds = [0] * 6
    volume_node.GetBounds(bounds)
    # bounds: [xmin, xmax, ymin, ymax, zmin, zmax]

    center_x = (bounds[0] + bounds[1]) / 2
    center_y = (bounds[2] + bounds[3]) / 2
    center_z = (bounds[4] + bounds[5]) / 2

    size_x = bounds[1] - bounds[0]
    size_y = bounds[3] - bounds[2]
    size_z = bounds[5] - bounds[4]

    # ROI: full width in X,Y; central 80% in Z (top and bottom 10% cut off)
    roi_node = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLMarkupsROINode", "LungROI"
    )
    roi_node.SetCenter(center_x, center_y, center_z)
    roi_node.SetSize(size_x, size_y, size_z * 0.8)

    # Display settings
    roi_node.GetDisplayNode().SetSelectedColor(0.0, 1.0, 0.0)
    roi_node.GetDisplayNode().SetHandleVisibility(True)
    roi_node.GetDisplayNode().SetFillVisibility(True)
    roi_node.GetDisplayNode().SetFillOpacity(0.1)

    print(f"  ROI created: center=({center_x:.0f}, {center_y:.0f}, {center_z:.0f})")
    print(f"  ROI size: {size_x:.0f} x {size_y:.0f} x {size_z*0.8:.0f} mm")

    return roi_node


# ============================================================
# Crop with CropVolume
# ============================================================
def crop_volume_with_roi(volume_node, roi_node):
    """Crop the volume with the ROI and return the new volume"""
    crop_params = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLCropVolumeParametersNode"
    )
    crop_params.SetInputVolumeNodeID(volume_node.GetID())
    crop_params.SetROINodeID(roi_node.GetID())
    crop_params.SetVoxelBased(True)
    crop_params.SetInterpolationMode(
        crop_params.InterpolationNearestNeighbor
    )

    slicer.modules.cropvolume.logic().Apply(crop_params)
    slicer.app.processEvents()

    cropped_node = slicer.mrmlScene.GetNodeByID(
        crop_params.GetOutputVolumeNodeID()
    )
    slicer.mrmlScene.RemoveNode(crop_params)

    if cropped_node:
        arr = slicer.util.arrayFromVolume(cropped_node)
        print(f"  Cropped volume: {arr.shape} voxels")

    return cropped_node


# ============================================================
# GMM analysis (extension or fallback)
# ============================================================
def run_gmm_analysis(volume_node, use_extension=True):
    """Run GMM-based lung tissue classification"""
    if use_extension:
        slicer.util.selectModule("LungCTGMMSegmentation")
        slicer.app.processEvents()

        module = slicer.modules.lungctgmmsegmentation
        widget = module.widgetRepresentation().self()

        output_seg = slicer.mrmlScene.AddNewNodeByClass(
            "vtkMRMLSegmentationNode", "GMM_Result"
        )
        averaged_seg = slicer.mrmlScene.AddNewNodeByClass(
            "vtkMRMLSegmentationNode", "GMM_Averaged"
        )

        widget.ui.inputVolumeSelector.setCurrentNode(volume_node)
        widget.ui.outputSegmentationSelector.setCurrentNode(output_seg)
        widget.ui.outputAveragedSegmentationSelector.setCurrentNode(averaged_seg)

        widget.onApplyButton()
        slicer.app.processEvents()
        time.sleep(2)

        return output_seg
    else:
        return run_fallback_gmm(volume_node)


def run_fallback_gmm(volume_node):
    """Re-fit the GMM with the paper's parameters and apply it"""
    from sklearn.mixture import GaussianMixture
    from scipy import ndimage
    from scipy.ndimage import median_filter

    volume_array = slicer.util.arrayFromVolume(volume_node)
    print(f"  Volume shape: {volume_array.shape}")

    # Lung region segmentation
    print("  1/6 Threshold (-155 HU)...")
    lung_mask = volume_array < -155

    print("  2/6 Island removal...")
    labeled_array, num_features = ndimage.label(lung_mask)
    if num_features > 0:
        sizes = ndimage.sum(lung_mask, labeled_array, range(1, num_features + 1))
        if len(sizes) > 2:
            threshold = sorted(sizes, reverse=True)[1] * 0.1
            for i, size in enumerate(sizes):
                if size < threshold:
                    lung_mask[labeled_array == (i + 1)] = False

    print("  3/6 Morphological closing...")
    struct = ndimage.generate_binary_structure(3, 2)
    lung_mask = ndimage.binary_closing(lung_mask, structure=struct, iterations=3)
    lung_mask = ndimage.binary_fill_holes(lung_mask)
    print(f"      Lung voxels: {np.sum(lung_mask):,}")

    print("  4/6 GMM fitting (5 components)...")
    lung_intensities = volume_array[lung_mask].astype(np.float64).reshape(-1, 1)
    gmm = GaussianMixture(
        n_components=5, covariance_type='full', n_init=6,
        init_params='kmeans', random_state=42, max_iter=200, tol=1e-3
    )
    gmm.fit(lung_intensities)
    print(f"      Converged: {gmm.converged_}")

    print("  5/6 MAP estimation...")
    raw_labels = gmm.predict(lung_intensities)
    means = gmm.means_.flatten()
    sorted_indices = np.argsort(means)
    label_mapping = {old: new for new, old in enumerate(sorted_indices)}
    mapped_labels = np.array([label_mapping[l] for l in raw_labels])

    label_map = np.zeros(volume_array.shape, dtype=np.int8)
    label_map[lung_mask] = mapped_labels + 1

    tissue_names = ["Air", "Healthy_Lung", "GGO", "Consolidation", "Dense_Tissue"]
    sorted_means = means[sorted_indices]
    for i, (name, mean) in enumerate(zip(tissue_names, sorted_means)):
        print(f"      {i}: {name} = {mean:.1f} HU")

    print("  6/6 Median filter...")
    mask_nz = label_map > 0
    filtered = median_filter(label_map, size=4)
    label_map[mask_nz] = filtered[mask_nz]

    # Create the segmentation node
    colors = {1:[0,0,0], 2:[0,0.5,1], 3:[1,1,0], 4:[1,0.5,0], 5:[1,0,0]}

    labelmapNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")
    labelmapNode.CopyOrientation(volume_node)
    slicer.util.updateVolumeFromArray(labelmapNode, label_map)
    labelmapNode.SetOrigin(volume_node.GetOrigin())
    labelmapNode.SetSpacing(volume_node.GetSpacing())
    ijkToRas = slicer.vtkMatrix4x4()
    volume_node.GetIJKToRASMatrix(ijkToRas)
    labelmapNode.SetIJKToRASMatrix(ijkToRas)

    output_seg = slicer.mrmlScene.AddNewNodeByClass(
        "vtkMRMLSegmentationNode", "GMM_Result"
    )
    slicer.modules.segmentations.logic().ImportLabelmapToSegmentationNode(
        labelmapNode, output_seg
    )
    segmentation = output_seg.GetSegmentation()
    for i in range(segmentation.GetNumberOfSegments()):
        seg_id = segmentation.GetNthSegmentID(i)
        segment = segmentation.GetSegment(seg_id)
        lbl = i + 1
        if lbl <= len(tissue_names):
            segment.SetName(tissue_names[lbl - 1])
        if lbl in colors:
            segment.SetColor(*colors[lbl])
    slicer.mrmlScene.RemoveNode(labelmapNode)

    output_seg.CreateDefaultDisplayNodes()
    output_seg.GetDisplayNode().SetVisibility2D(True)

    return output_seg


# ============================================================
# Segment Statistics
# ============================================================
def compute_and_save(volume_node, seg_node, patient_name, output_dir):
    """Compute statistics, display results, save files, and return result_row"""
    import SegmentStatistics

    segStatLogic = SegmentStatistics.SegmentStatisticsLogic()
    segStatLogic.getParameterNode().SetParameter("Segmentation", seg_node.GetID())
    segStatLogic.getParameterNode().SetParameter("ScalarVolume", volume_node.GetID())
    segStatLogic.computeStatistics()
    stats = segStatLogic.getStatistics()

    result_row = {"patient": patient_name}
    total_vol = 0

    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = stats[segId, "LabelmapSegmentStatisticsPlugin.volume_cm3"]
        result_row[f"{seg_name}_cm3"] = round(vol, 2)
        total_vol += vol

    result_row["total_cm3"] = round(total_vol, 2)

    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = stats[segId, "LabelmapSegmentStatisticsPlugin.volume_cm3"]
        pct = round(vol / total_vol * 100, 2) if total_vol > 0 else 0
        result_row[f"{seg_name}_pct"] = pct

    # Lung involvement
    ggo = consolidation = dense = healthy = 0
    for key, val in result_row.items():
        kl = key.lower()
        if key.endswith("_pct"):
            if "glass" in kl or "ggo" in kl:
                ggo = val
            elif "consolidat" in kl:
                consolidation = val
            elif "dense" in kl:
                dense = val
            elif "healthy" in kl:
                healthy = val

    result_row["lung_involvement_pct"] = round(ggo + consolidation + dense, 2)

    # Display
    print(f"\n  --- Results: {patient_name} ---")
    for segId in stats["SegmentIDs"]:
        seg_name = stats[segId, "Segment"]
        vol = result_row.get(f"{seg_name}_cm3", 0)
        pct = result_row.get(f"{seg_name}_pct", 0)
        bar = '#' * int(pct / 2)
        print(f"  {seg_name:20s}: {vol:8.1f} cm3 ({pct:5.1f}%) {bar}")
    print(f"  {'Total':20s}: {total_vol:8.1f} cm3")
    print(f"  Lung involvement (I) = {result_row['lung_involvement_pct']:.1f}%")
    print(f"  Healthy lung         = {healthy:.1f}%")

    # Save
    pt_dir = os.path.join(output_dir, patient_name)
    os.makedirs(pt_dir, exist_ok=True)

    slicer.util.saveNode(seg_node, os.path.join(pt_dir, "gmm_segmentation.seg.nrrd"))
    segStatLogic.exportToCSVFile(os.path.join(pt_dir, "segment_statistics.csv"))

    json_data = {
        "method": "Zaffino_et_al_2021_GMM",
        "patient": patient_name,
        "analysis_date": datetime.now().isoformat(),
        "slicer_version": slicer.app.applicationVersion,
        "roi_cropped": True,
        "lung_involvement_pct": result_row["lung_involvement_pct"],
        "results": {k: v for k, v in result_row.items() if k != "patient"}
    }
    with open(os.path.join(pt_dir, "gmm_results.json"), 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

    print(f"  Saved to: {pt_dir}")
    return result_row


print("Helper functions loaded.")

---
## 3. Load the patient list

In [ ]:
# Initialize the DICOM database
ensure_dicom_database()

# Get the patient list
patients = find_dicom_patients(INPUT_DIR)
print(f"Found {len(patients)} patients:\n")
for i, p in enumerate(patients):
    print(f"  {i+1:3d}. {p['name']}  ({p['n_slices']} slices)")

# Check for the extension
use_ext = not FALLBACK_MODE
if use_ext:
    try:
        _ = slicer.modules.lungctgmmsegmentation
        print(f"\nSlicerDensityLungSegmentation: detected")
    except AttributeError:
        print(f"\nExtension not found -> fallback mode")
        use_ext = False

# Progress tracking
current_index = 0
all_results = []
skipped = []

# Variables holding the current state
current_volume = None
current_roi = None
current_patient = None

print(f"\n--- Run Cell 4 to load the first patient ---")

---
## 4. Load the next patient + create ROI

**When you run this cell:**
1. It loads the next patient's DICOM
2. The Slicer view shows the CT and a **green ROI box**

**Once it appears, in the Slicer view:**
- **Drag the ROI box handles (the round points on corners and edges)** to enclose only the lung region
- Adjust the **Z axis (up/down)** in particular to exclude the neck and abdomen
- If the trachea is included, lower the top edge

**When you are done adjusting, run Cell 5**

In [ ]:
# === Load the next patient ===

if current_index >= len(patients):
    print("All patients have been processed.")
    print("Run Cell 6 to export the aggregated CSV.")
else:
    current_patient = patients[current_index]
    print(f"{'='*60}")
    print(f"[{current_index + 1}/{len(patients)}] {current_patient['name']}")
    print(f"{'='*60}")

    # Clear the scene
    slicer.mrmlScene.Clear(0)
    slicer.app.processEvents()
    time.sleep(1)

    # Load DICOM
    print("Loading DICOM...")
    current_volume = load_volume_from_dicom(current_patient["path"])

    if current_volume:
        # Fit the slice views
        slicer.util.setSliceViewerLayers(background=current_volume)
        slicer.util.resetSliceViews()
        slicer.app.processEvents()

        # Create ROI
        current_roi = create_lung_roi(current_volume)
        slicer.app.processEvents()

        print(f"\n>> Adjust the ROI to the lung region in the Slicer view")
        print(f">> After adjusting, run Cell 5")
        print(f">> To skip, re-run Cell 4 instead of Cell 5")
    else:
        print(f"  ERROR: failed to load the volume")
        skipped.append(current_patient["name"])
        current_index += 1
        print(f"  Re-run Cell 4 to move on to the next patient")

---
## 5. Crop with ROI -> GMM analysis -> save

**Run this cell only after you have finished adjusting the ROI.**

What it does:
1. Crop the volume with the ROI
2. Run the GMM analysis on the cropped volume
3. Display and save the results
4. Automatically move on to the next patient (go back and run Cell 4)

In [ ]:
# === Crop with ROI -> GMM analysis -> save ===

if current_volume is None or current_roi is None:
    print("ERROR: run Cell 4 first to load a patient")
elif current_patient is None:
    print("ERROR: no patient data")
else:
    patient_name = current_patient["name"]
    print(f"Processing: {patient_name}")
    t0 = time.time()

    try:
        # 1. Crop with ROI
        print("\n  [1/3] Cropping with ROI...")
        cropped_volume = crop_volume_with_roi(current_volume, current_roi)

        if cropped_volume is None:
            raise RuntimeError("Crop failed")

        # 2. GMM analysis
        print("\n  [2/3] Running GMM analysis...")
        seg_node = run_gmm_analysis(cropped_volume, use_extension=use_ext)

        if seg_node is None:
            raise RuntimeError("GMM analysis failed")

        # 3. Compute statistics + save
        print("\n  [3/3] Computing statistics...")
        result_row = compute_and_save(
            cropped_volume, seg_node, patient_name, OUTPUT_DIR
        )
        all_results.append(result_row)

        elapsed = time.time() - t0
        print(f"\n  Time: {elapsed:.1f}s")

    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback
        traceback.print_exc()
        skipped.append(patient_name)

    # Move on to the next patient
    current_index += 1
    current_volume = None
    current_roi = None
    current_patient = None

    print(f"\n{'='*60}")
    print(f"Progress: {current_index}/{len(patients)} "
          f"(completed: {len(all_results)}, skipped: {len(skipped)})")
    if current_index < len(patients):
        print(f"Next: {patients[current_index]['name']}")
        print(f"--- Run Cell 4 to load the next patient ---")
    else:
        print(f"\nAll patients have been processed!")
        print(f"--- Run Cell 6 to export the aggregated CSV ---")
    print(f"{'='*60}")

---
## 6. Export the aggregated results

Run this after all cases have been processed. All results are aggregated into `batch_summary.csv`.

In [ ]:
# === Export aggregated CSV ===

print(f"{'='*60}")
print(f"BATCH SUMMARY")
print(f"{'='*60}")
print(f"  Completed: {len(all_results)}/{len(patients)}")
if skipped:
    print(f"  Skipped/Failed: {len(skipped)} ({', '.join(skipped)})")

if all_results:
    # CSV export
    summary_path = os.path.join(OUTPUT_DIR, "batch_summary.csv")
    fieldnames = list(dict.fromkeys(
        key for r in all_results for key in r.keys()
    ))

    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_results)

    print(f"\n  Summary CSV: {summary_path}")

    # Results listing
    print(f"\n  {'Patient':15s} {'Involvement':>12s} {'Healthy':>10s} {'GGO':>8s} {'Consol.':>8s} {'Dense':>8s}")
    print(f"  {'-'*65}")
    for r in all_results:
        inv = r.get('lung_involvement_pct', 0)
        # Get the fraction for each tissue (handles differing key names)
        healthy = ggo = consol = dense = 0
        for k, v in r.items():
            kl = k.lower()
            if k.endswith('_pct'):
                if 'healthy' in kl:
                    healthy = v
                elif 'glass' in kl or 'ggo' in kl:
                    ggo = v
                elif 'consolidat' in kl:
                    consol = v
                elif 'dense' in kl:
                    dense = v
        print(f"  {r['patient']:15s} {inv:11.1f}% {healthy:9.1f}% {ggo:7.1f}% {consol:7.1f}% {dense:7.1f}%")

    print(f"\n{'='*60}")

    # Open in Finder
    import subprocess
    subprocess.call(['open', OUTPUT_DIR])
else:
    print("\n  No results to export.")

---
## Appendix: Reprocess specific patients only

To reprocess a specific patient, change `current_index` in the cell below, then run Cell 4 -> 5.

In [ ]:
# Jump to a specific patient (check the number in the Cell 3 listing)
# Example: to jump to the 5th patient
# current_index = 4  # 0-based, so subtract 1

# To search by patient name
# target = "03309523-1"
# current_index = next(i for i, p in enumerate(patients) if p["name"] == target)

print(f"Current index: {current_index}")
if current_index < len(patients):
    print(f"Next patient: {patients[current_index]['name']}")
print(f"\nTo change: uncomment and edit the lines above, then run Cell 4")